#1. Load Data

In [3]:
%%capture
!pip install kagglehub
!pip install swifter

Load from kaggle

In [4]:
import warnings
warnings.filterwarnings("ignore")
import kagglehub
import numpy as np
import pandas as pd
import regex as re
from swifter import swifter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.cuda.amp import autocast, GradScaler

torch.cuda.empty_cache()
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
dir = kagglehub.dataset_download('gowrishankarp/newspaper-text-summarization-cnn-dailymail')

In [8]:
train = pd.read_csv(f'{dir}/cnn_dailymail/train.csv')
val = pd.read_csv(f'{dir}/cnn_dailymail/validation.csv')

In [9]:
train.drop('id', axis=1, inplace=True)
val.drop('id', axis=1, inplace=True)

In [10]:
train.head()

,article,highlights
0,By . Associated Press . PUBLISHED: . 14:11 EST...,"Bishop John Folda, of North Dakota, is taking ..."
1,(CNN) -- Ralph Mata was an internal affairs li...,Criminal complaint: Cop used his role to help ...
2,A drunk driver who killed a young woman in a h...,"Craig Eccleston-Todd, 27, had drunk at least t..."
3,(CNN) -- With a breezy sweep of his pen Presid...,Nina dos Santos says Europe must be ready to a...
4,Fleetwood are the only team still to have a 10...,Fleetwood top of League One after 2-0 win at S...


In [11]:
train.shape

(287113, 2)

In [12]:
train = train[:40000]

In [13]:
train.shape

(40000, 2)

# 2. Cleanin/Preprocessing

In [19]:
import re #regex
def clean_data(data):
  data = re.sub('\s+\n+', ' ', data)
  data = data.lower()
  data = re.sub('[^a-zA-Z0-9\.,:]', ' ', data)
  data = data.split()
  data = ['<s>'] + [word for word in data] + ['</s>']
  data = ' '.join(data)
  return data


In [20]:
train['article'] = train['article'].swifter.apply(clean_data)
train['highlights'] = train['highlights'].swifter.apply(clean_data)

val['article'] = val['article'].swifter.apply(clean_data)
val['highlights'] = val['highlights'].swifter.apply(clean_data)

Pandas Apply:   0%|          | 0/40000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/40000 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/13368 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/13368 [00:00<?, ?it/s]

In [21]:
train['article'][0]

'<s> by . associated press . published: . 14:11 est, 25 october 2013 . . updated: . 15:36 est, 25 october 2013 . the bishop of the fargo catholic diocese in north dakota has exposed potentially hundreds of church members in fargo, grand forks and jamestown to the hepatitis a virus in late september and early october. the state health department has issued an advisory of exposure for anyone who attended five churches and took communion. bishop john folda pictured of the fargo catholic diocese in north dakota has exposed potentially hundreds of church members in fargo, grand forks and jamestown to the hepatitis a . state immunization program manager molly howell says the risk is low, but officials feel it s important to alert people to the possible exposure. the diocese announced on monday that bishop john folda is taking time off after being diagnosed with hepatitis a. the diocese says he contracted the infection through contaminated food while attending a conference for newly ordained 

In [22]:
train["highlights"][0]

'<s> bishop john folda, of north dakota, is taking time off after being diagnosed . he contracted the infection through contaminated food in italy . church members in fargo, grand forks and jamestown could have been exposed . </s>'

# 3. Modeling

In [28]:
from collections import Counter

#tokenize
def extract_token(data):
  return [token for text in data for token in text.split()]

#special chars
special_token_input = ['<pad>', '<unk>', '<s>', '</s>']
special_token_output = ['<pad>', '<s>', '</s>']

#extract and count token
article_tokens = extract_token(train['article'])
summary_tokens = extract_token(train['highlights'])

input_counter = Counter(article_tokens)
output_counter = Counter(summary_tokens)

#build a vocab of items that appear more twice
input_vocab = special_token_input + [token for token, freq in input_counter.items() if freq >= 2]
output_vocab = special_token_output + [token for token, freq in output_counter.items() if freq >=2]

#word2idx
text_vocab = {word: idx for idx, word in enumerate(input_vocab)}
summary_vocab = {word: idx for idx, word in enumerate(output_vocab)}
idx_to_word = {idx: word for word, idx in summary_vocab.items()}




In [29]:
print(f"Input vocab size: {len(text_vocab)}")
print(f"Output vocab size: {len(summary_vocab)}")

# Spot check mappings
print(text_vocab.get('<unk>'))           # special token
print(idx_to_word[summary_vocab['</s>']])    # reverse mapping check

Input vocab size: 196779
Output vocab size: 44367
1
</s>


In [136]:
input_dim = len(input_vocab)
output_dim = len(output_vocab)
embedding_dim = 32
hidden_dim = 32
LR = 0.005
batch_size = 8
epochs = 3

Dataset Loader

In [177]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch

class CNNDailyMailDataset(Dataset):
  def __init__(self, data):
    self.data = data

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
        src = torch.tensor(list(map(lambda x: text_vocab.get(x, text_vocab['<unk>']), (self.data.loc[idx, "article"].split()))))
        tgt = torch.tensor(list(map(lambda x: summary_vocab.get(x, summary_vocab['<pad>']), (self.data.loc[idx, "highlights"].split()))))
        return src, tgt

def collate_fn(batch):
  src_batch, tgt_batch = zip(*batch)

  src_padded = pad_sequence(src_batch, batch_first=True, padding_value=text_vocab['<pad>'])
  tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=summary_vocab['<pad>'])

  return src_padded, tgt_padded



train_dataset = CNNDailyMailDataset(train)
val_dataset = CNNDailyMailDataset(val)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=True
)

Encoder Implementation

In [178]:
import torch.nn as nn

class BLSTMEncoder(nn.Module):
  def __init__(self, input_dim, embedding_dim, hidden_dim, num_layers=1):
    super().__init__()
    self.embedding = nn.Embedding(input_dim, embedding_dim)
    self.lstm = nn.LSTM(
        input_size = embedding_dim ,
        hidden_size=hidden_dim,
        num_layers=1,
        bidirectional=True,
        batch_first=True
      )
    self.hidden_dim = hidden_dim
    self.num_layers = num_layers
  def forward(self, src):
    embedded = self.embedding(src.long())
    outputs, (hidden_state, cell_state) = self.lstm(embedded)

    return outputs, (hidden_state, cell_state)

Decoder implementation

In [179]:
class LSTMDecoder(nn.Module):
  def __init__(self, output_dim, embedding_dim, hidden_dim):
    super().__init__()
    self.embedding = nn.Embedding(output_dim, embedding_dim)
    self.lstm = nn.LSTM(
        embedding_dim +hidden_dim * 2,
        hidden_dim,
        batch_first=True
    )
    self.fc_out = nn.Linear(hidden_dim, output_dim)
    self.attention = Attention(hidden_dim)
    self.output_dim = output_dim
    self.hidden_dim = hidden_dim

  def forward(self, input, hidden, cell, encoder_outputs, mask):
    embedded = self.embedding(input)

    attn_weights = self.attention(hidden[-1], encoder_outputs, mask)
    attn_weights = attn_weights.unsqueeze(1)

    context = torch.bmm(attn_weights, encoder_outputs)
    lstm_input = torch.cat(
        (embedded.unsqueeze(1), context),
        dim=2
    )

    output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))
    prediction = self.fc_out(output.squeeze(1))

    return prediction, hidden, cell

Attention module

In [180]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Attention(nn.Module):
  def __init__(self, hidden_dim):
    super().__init__()
    self.attn = nn.Linear(hidden_dim * 3, hidden_dim)
    self.v = nn.Linear(hidden_dim, 1, bias=False)

  def forward(self, hidden, encoder_outputs, mask):
    batch_size, src_len, _ = encoder_outputs.size()
    hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
    energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
    scores = self.v(energy).squeeze(2)
    scores = scores.masked_fill(~mask, -1e9)
    return F.softmax(scores, dim=1)

def create_mask(src, pad_idx):
  return src != pad_idx

Seq to seq wrapper

In [212]:
import torch
import torch.nn as nn
import random

class Seq2Seq(nn.Module):
    def __init__(self, encoder: BLSTMEncoder, decoder: LSTMDecoder, src_pad_idx: int, device: torch.device):
        super().__init__()
        self.encoder      = encoder
        self.decoder      = decoder
        self.src_pad_idx  = src_pad_idx
        self.device       = device

        self.fc_hidden = nn.Linear(encoder.hidden_dim * 2, decoder.hidden_dim)
        self.fc_cell   = nn.Linear(encoder.hidden_dim * 2, decoder.hidden_dim)

    def create_mask(self, src):
        return src != self.src_pad_idx

    def forward(self, src, trg, teacher_forcing_ratio: float = 0.5):


        batch_size, trg_len = trg.size()
        output_dim = self.decoder.output_dim

        # tensor to store all decoder outputs
        outputs = torch.zeros(batch_size, trg_len, output_dim).to(self.device)

        #Encoder
        encoder_outputs, (enc_hidden, enc_cell) = self.encoder(src)
        mask = self.create_mask(src)

        #Bidir to unidir initial hidden/cell
        forward_h = enc_hidden[-2,:,:]
        backward_h= enc_hidden[-1,:,:]
        hidden = torch.tanh(self.fc_hidden(torch.cat((forward_h, backward_h), dim=1)))
        hidden = hidden.unsqueeze(0)

        forward_c = enc_cell[-2,:,:]
        backward_c= enc_cell[-1,:,:]
        cell = torch.tanh(self.fc_cell(torch.cat((forward_c, backward_c), dim=1)))
        cell = cell.unsqueeze(0)

        #Decoder loop
        input_tok = trg[:,0]

        for t in range(1, trg_len):
            #decode one step
            output, hidden, cell = self.decoder(
                input_tok, hidden, cell,
                encoder_outputs, mask
            )
            outputs[:,t] = output

            #teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input_tok = trg[:,t] if teacher_force else top1

        return outputs


In [182]:

SRC_PAD_IDX = text_vocab['<pad>']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

enc = BLSTMEncoder(input_dim, embedding_dim, hidden_dim, num_layers=1)
dec = LSTMDecoder(output_dim, embedding_dim, hidden_dim)

# supply pad‐index and device, then move to device
model = Seq2Seq(enc, dec, src_pad_idx=SRC_PAD_IDX, device=device).to(device)

optimizer = optim.AdamW(model.parameters(), lr=LR)
# ignore the <pad> token in the loss
criterion = nn.CrossEntropyLoss(ignore_index=summary_vocab['<pad>'])


In [183]:
import sys
import torch

def run_epoch(model, loader, optimizer, criterion, device, training=True):
    model.train() if training else model.eval()
    total_loss = 0.0
    total_batches = len(loader)

    for batch_idx, (source, target) in enumerate(loader, 1):
        # move to GPU/CPU
        source = source.to(device)
        target = target.to(device)

        if training:
            optimizer.zero_grad()
            output = model(source, target)

        else:
            with torch.no_grad():
                output = model(source, target, teacher_forcing_ratio=0.0)

        # skip <sos> step when computing loss
        # slice along time-axis, then flatten
        vocab_size = output.size(-1)
        pred_tokens = output[:, 1:, :].contiguous().view(-1, vocab_size)
        gold_tokens = target[:, 1:].contiguous().view(-1)

        loss = criterion(pred_tokens, gold_tokens)

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

        #print batch-level progress
        avg_so_far = total_loss / batch_idx
        sys.stdout.write(
            f'\r{batch_idx}/{total_batches} ' +
            f'{"Train" if training else "Valid"} Loss: {avg_so_far:.4f}'
        )
        sys.stdout.flush()

    print()
    return total_loss / total_batches




for epoch in range(1, epochs + 1):
    print(f"Epoch {epoch}/{epochs}")
    train_loss = run_epoch(model, train_loader, optimizer, criterion, device, training=True)
    val_loss   = run_epoch(model,   val_loader,   optimizer, criterion, device, training=False)
    print(f"▸ Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}\n")


Epoch 1/3
5000/5000 Train Loss: 7.1802
1671/1671 Valid Loss: 7.0789
▸ Train Loss: 7.1802 | Val Loss: 7.0789

Epoch 2/3
5000/5000 Train Loss: 6.7181
1671/1671 Valid Loss: 6.9874
▸ Train Loss: 6.7181 | Val Loss: 6.9874

Epoch 3/3
5000/5000 Train Loss: 6.5172
1671/1671 Valid Loss: 6.9525
▸ Train Loss: 6.5172 | Val Loss: 6.9525



In [184]:
save_path = "seq2seq_checkpoint.pt"
torch.save({
    'epoch': epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_loss,
    'val_loss': val_loss,
}, save_path)
print(f"Saved checkpoint to {save_path}")

Saved checkpoint to seq2seq_checkpoint.pt


In [185]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [187]:
save_path = "/content/drive/MyDrive/seq2seq_checkpoint.pt"
torch.save({
    'epoch': epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_loss,
    'val_loss': val_loss,
}, save_path)
print(f" Saved checkpoint to {save_path}")

 Saved checkpoint to /content/drive/MyDrive/seq2seq_checkpoint.pt


In [192]:
test = pd.read_csv(f'{dir}/cnn_dailymail/test.csv')
test.drop('id', axis=1, inplace=True)

test['article'] = test['article'].swifter.apply(clean_data)
test['highlights'] = test['highlights'].swifter.apply(clean_data)

Pandas Apply:   0%|          | 0/11490 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/11490 [00:00<?, ?it/s]

# TEST

In [201]:
import torch

def summary(model, article_text, max_len=50):
    """
    Generate an abstractive summary
    """
    model.eval()

    Tokenize & index
    tokens = clean_data(article_text).split()
    indexed = [text_vocab.get(tok, text_vocab['<unk>']) for tok in tokens]
    src = torch.LongTensor(indexed).unsqueeze(0).to(device)  # [1, src_len]

    with torch.no_grad():
        #Encode
        enc_outputs, (h_n, c_n) = model.encoder(src)
        mask = create_mask(src, SRC_PAD_IDX)

        #Initialize decoder state by projecting final forward/backward encoder states
        fwd_h, bwd_h = h_n[-2], h_n[-1]
        dec_h = torch.tanh(model.fc_hidden(torch.cat((fwd_h, bwd_h), dim=1)))
        dec_h = dec_h.unsqueeze(0)

        fwd_c, bwd_c = c_n[-2], c_n[-1]
        dec_c = torch.tanh(model.fc_cell(torch.cat((fwd_c, bwd_c), dim=1)))
        dec_c = dec_c.unsqueeze(0)

        #Iteratively decode
        input_tok = torch.LongTensor([summary_vocab['<s>']]).to(device)
        output_indices = []

        for _ in range(max_len):
            logits, dec_h, dec_c = model.decoder(input_tok, dec_h, dec_c, enc_outputs, mask)
            top1 = logits.argmax(1).item()

            if top1 == summary_vocab['</s>']:
                break

            output_indices.append(top1)
            input_tok = torch.LongTensor([top1]).to(device)

    #Convert indices back to words
    summary_words = [idx_to_word[idx] for idx in output_indices]
    return ' '.join(summary_words)



In [208]:
print(test.loc[9, 'article'])
summary(model, test.loc[9, 'article'])

<s> a gang of six men have been jailed for a total of 31 years after being convicted of a string of sexual offences against teenager girls. the offences, which ranged from inciting sexual activity with a child, to rape, happened in cars, woods or at the defendants homes in banbury, oxfordshire. oxford crown court heard how they lured victims to parties organised on social media and then began sexually abusing them. the men were found guilty in march and have now been handed sentences of between three and nine years in jail. jailed: ahmed hassan sule, 21 left was sentenced to nine years imprisonment. mohamed saleh, 22, right was imprisoned for four years and nine months . the girls, aged between 13 and 16, were targeted by the gang at under 18s parties organised by ahmed hassan sule, 21, known as fiddy . one child described the parties as a place where girls would go and the boys would choose their targets . the victims, who were described in court as emotionally immature were abused fr

'police say found guilty to old year old year old year old was in the . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .'

With my computational limitations I wasnt able to train the model well enough. The output summary from the model is giberish and its a result of the model not learning a good signal

Ive still decided to implement the BART version to test out summarization but i will not be comparing rouge scores being that the LSTM model was not able to produce a decent output

# BART

In [210]:
!pip install datasets
from datasets import load_dataset

# Load DATA
dataset = load_dataset("cnn_dailymail", "3.0.0")
print(dataset)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [218]:
from transformers import BartTokenizer, BartForConditionalGeneration
from datasets import load_dataset
import torch

# Load tokenizer & model
tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn").to("cuda" if torch.cuda.is_available() else "cpu")

# Load the CNN/DailyMail dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Pick a sample article
sample = dataset["test"][0]
article = sample["article"]
reference_summary = sample["highlights"]

# Tokenize & generate
inputs = tokenizer(article, return_tensors="pt", max_length=1024, truncation=True).to(model.device)
summary_ids = model.generate(inputs["input_ids"], num_beams=4, max_length=128, early_stopping=True)
generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Print everything
print("=" * 80)
print("Original Article:")
print(article)
print("\n" + "=" * 80)
print("Reference Summary (Ground Truth):")
print(reference_summary)
print("\n" + "=" * 80)
print("BART-Generated Summary:")
print(generated_summary)


Original Article:
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednes

As you can see with BART already being finetuned on this dataset and good at zero shot summarization it performs well on this data